# early_stopping

Generated with **Lispyn** — originally created by **Eduardo J. Barrios**.

Source: [Lispyn](https://github.com/edujbarrios/lispyn) (Apache-2.0).

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='early_stopping', assignments=[['md', 'md']], prefix='')


# Validation-guided training

Define an early-stopping policy in Lisp. Checkpoints and real metrics stay available for inspection.

In [ ]:
import hy
import torch
hy.macros.require('lispyn.macros', None, target_module_name='early_stopping', assignments=[['with-seed', 'with-seed']], prefix='')
from lispyn.reproducibility import seed_all
seed_all(42)
model = torch.nn.Sequential(torch.nn.Linear(2, 8), torch.nn.ReLU(), torch.nn.Linear(8, 2))
x = torch.tensor([[0.0, 0.0], [0.2, 0.1], [0.9, 0.1], [0.8, 0.2], [0.1, 0.9], [0.2, 0.8], [1.0, 1.0], [0.9, 0.8], [0.0, 0.2], [0.9, 0.0], [0.0, 1.0], [1.0, 0.9]])
y = torch.tensor([0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1])
train_data = torch.utils.data.TensorDataset(x[slice(0, 8)], y[slice(0, 8)])
validation = torch.utils.data.TensorDataset(x[slice(8, None)], y[slice(8, None)])


## Stop when validation stops improving

Train for at most 40 epochs. After four epochs without a validation-loss improvement greater than 0.01, stop. The best and last checkpoints are saved separately.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='early_stopping', assignments=[['train-until', 'train-until']], prefix='')
from lispyn.workflow import train
run = train(model, train_data, patience=4, validation=validation, loss_fn=torch.nn.CrossEntropyLoss(), epochs=40, min_delta=0.01, metrics='accuracy', batch_size=4, lr=0.02, device='cpu', seed=42, experiment_dir='runs/early-stop')
print('Completed epochs:', len(run.history['train_loss']))
print('Stopped early:', run.stopped_early)


## Present the results

Render the recorded training and validation losses in the shared Docusaurus-dark turquoise design.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='early_stopping', assignments=[['defcurve', 'defcurve']], prefix='')
from lispyn.plots import make_plotter
training_chart = make_plotter(theme='docusaurus-dark', palette='turquoise-spectrum', train_color='#40E0D0', val_color='#8CF0EB')
training_chart(run, metric='loss', title='Validation-guided training', subtitle='Early stopping · Tiny offline classifier', save='runs/early-stop/loss.png')
